# Visualize UniDepth-LSS

Renders one random validation sample: the six camera views, the ground-truth BEV vehicle map and
the model's prediction. Ground truth is blue, the raw thresholded prediction is red, roads are
grey and the ego vehicle is white.

Run it from inside the container (`make jupyter`). Outputs are not committed -- clear them before
saving, or the notebook grows by megabytes per run.


In [ ]:
import os
from pathlib import Path

import rootutils
import torch

rootutils.setup_root(os.getcwd(), indicator=".project-root", pythonpath=True)

from unidepthlss.data.dataset import NuScenesBEVDataset
from unidepthlss.geometry import BevGrid
from unidepthlss.modeling.model import UniDepthLSS
from unidepthlss.utils.visualization import visualize_random_sample

# Same env-var-with-container-fallback convention as configs/data/nuscenes.yaml.
DATA_ROOT = Path(os.environ.get("NUSCENES_DATA_ROOT", "/data/nuscenes"))
CHECKPOINT = Path(os.environ.get("UNIDEPTHLSS_CHECKPOINT", "checkpoints/UniDepthLSS.pt"))

IMAGE_SIZE = (448, 798)

# The BEV window. `paper` reproduces the released figures; swap to the standard 100 x 100 m
# window (h=200, w=200, h_meters=100.0, w_meters=100.0) with depth_max=61.0 to see how the model
# behaves over the range the Table 1 baselines report on. See docs/baseline.md.
GRID = BevGrid(h=128, w=128, h_meters=64.0, w_meters=64.0)
DEPTH_MAX = 50.0

PREDICTION_THRESHOLD = 0.5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(GRID.describe())


In [ ]:
dataset = NuScenesBEVDataset(
    dataroot=str(DATA_ROOT),
    version="v1.0-trainval",
    split="val",
    img_size=IMAGE_SIZE,
    grid=GRID,
    augment=False,
)

model = UniDepthLSS(
    img_height=IMAGE_SIZE[0],
    img_width=IMAGE_SIZE[1],
    num_classes=1,
    feature_channels=128,
    grid=GRID,
    depth_max=DEPTH_MAX,
)

# Reuses tools/eval.py's loader: it handles the released checkpoint, a Lightning checkpoint and a
# bare state dict, and tolerates the absent backbone keys (those come from the HF hub, not the
# checkpoint) while still raising on any genuine mismatch.
from tools.eval import load_checkpoint_into

load_checkpoint_into(model, str(CHECKPOINT))
model = model.to(DEVICE).eval()


In [ ]:
figure, sample_index = visualize_random_sample(
    dataset, model, DEVICE, DATA_ROOT,
    threshold=PREDICTION_THRESHOLD, bev_resolution=GRID.res_h,
)
print(f"Visualized validation sample {sample_index}")
